In [17]:
import pandas as pd
import numpy as np

# Load the messy dataset
df = pd.read_csv('messy_netflix_titles.csv')
print("Initial missing values count:\n", df.isnull().sum())

Initial missing values count:
 SHOW ID            0
  Type           447
Title            447
Director        2680
Cast             838
COUNTRY         1245
DATE_ADDED        10
Release Year    2205
Rating           451
Duration         449
Listed In          0
Description        0
dtype: int64


In [18]:
# Strip whitespaces, lowercase everything, and use underscores for spaces
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [19]:
# 1. Standardize 'type' column strings
df['type'] = df['type'].astype(str).str.strip().str.title()
df['type'] = df['type'].replace({'Nan': np.nan, 'None': np.nan})

# 2. Standardize 'country' column strings 
def standardize_country(val):
    if pd.isna(val) or not isinstance(val, str) or val.strip().lower() in ['nan', 'none']:
        return np.nan
    val = val.strip().upper()
    if val in ['US', 'USA', 'UNITED STATES']:
        return 'United States'
    if val in ['IND', 'INDIA']:
        return 'India'
    return val.title()

df['country'] = df['country'].apply(standardize_country)

In [20]:
# CRITICAL FIX: Adding format='mixed' allows Pandas to parse multiple date structures correctly
df['date_added'] = pd.to_datetime(df['date_added'].astype(str).str.strip(), errors='coerce', format='mixed')

# Clean out extra words from release year and convert to numeric floating values temporarily
df['release_year'] = df['release_year'].astype(str).str.replace('Year:', '', case=False).str.strip()
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')

In [21]:
# Deduplicate using the unique identifier 'show_id'
df = df.drop_duplicates(subset=['show_id'], keep='first')
print(f"Dataset shape after deduplication: {df.shape}")

Dataset shape after deduplication: (8807, 12)


In [22]:
# 1. Handle primary fields
df['title'] = df['title'].fillna('Unknown Title')
df['type'] = df['type'].fillna('Unknown Type')

# 2. Handle missing categorical text fields with placeholder labels
df['director'] = df['director'].fillna('Unknown Director')
df['cast'] = df['cast'].fillna('Unknown Cast')
df['country'] = df['country'].fillna('Unknown Country')
df['rating'] = df['rating'].fillna('Not Rated')
df['duration'] = df['duration'].fillna('Unknown Duration')

# 3. Handle numerical fields by imputing the median year
median_year = int(df['release_year'].median())
df['release_year'] = df['release_year'].fillna(median_year).astype(int)

# 4. Finalize date formatting and fill unparseable/missing dates
df['date_added'] = df['date_added'].dt.strftime('%d-%m-%Y').fillna('Not Available')

In [23]:
# Check remaining null values (This will print 0 for everything!)
print("Final Missing Values Check:\n", df.isnull().sum())

# Save the pristine dataset
df.to_csv('cleaned_netflix_titles.csv', index=False)
print("SUCCESS: Full clean complete with zero missing values remaining!")

Final Missing Values Check:
 show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64
SUCCESS: Full clean complete with zero missing values remaining!
